# FinReasoningAI — Evaluation Notebook

Evaluates the fine-tuned Qwen2.5-14B model across three tasks:

| Section | Dataset | Conditions |
|---------|---------|------------|
| **A** | Synthetic test set (from Colab training) | Base model vs Fine-tuned |
| **B** | FinQA (≤300 samples) | Base/Fine-tuned × Direct/CoT |
| **C** | FinQA agentic | Fine-tuned + tools vs Fine-tuned (CoT only) |

All inference is done via **vLLM** with LoRA hot-swapping. Batch sizes are adjustable per dataset.

In [ ]:
# Step 0 — Install dependencies
!pip -q install vllm datasets pandas tqdm transformers peft

---
## Step 1 — Configuration
All tuneable parameters live here. Edit before running.

In [ ]:
# ============================================================
# CONFIGURATION — edit these before running the notebook
# ============================================================

# --- Google Drive / repo paths ---
DRIVE_BASE  = "/content/drive/MyDrive/FinReasoningAI"
REPO_URL    = "https://github.com/juankim834/FinReasoningAI.git"
REPO_DIR    = f"{DRIVE_BASE}/FinReasoningAI"
OUTPUTS_DIR = f"{DRIVE_BASE}/outputs"

# --- Processed DatasetDict produced by FinReasoningAI_Colab.ipynb ---
# Must match PROCESSED_DATA_DIR in the Colab notebook (default: "data/processed_fincot_sft").
# This directory is written by prepare_fincot_sft_dataset() and contains the
# stratified "train" and "test" splits used for SFT.
PROCESSED_DATA_DIR = "data/processed_fincot_sft"

# If the processed directory does not yet exist when you run Section A,
# the notebook will call prepare_fincot_sft_dataset() to regenerate it.
# These values MUST match the Colab training configuration exactly.
FINAL_SAMPLE_SIZE = 5000
TRAIN_SIZE        = 4500
TEST_SIZE         = 500
INCLUDE_COT       = True
SEED              = 42

# --- Fine-tuned adapter (LoRA weights from SFT training) ---
ADAPTER_PATH = f"{DRIVE_BASE}/outputs/sft_qlora/final_adapter"

# --- Model ---
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

# --- vLLM settings ---
GPU_MEMORY_UTILIZATION = 0.92
MAX_MODEL_LEN          = 8192   # vLLM context window; must satisfy:
#                                  MAX_PROMPT_TOKENS + max(MAX_TOKENS_COT, MAX_TOKENS_AGENTIC) < MAX_MODEL_LEN
LORA_RANK              = 64     # must match training config (r=64)

# --- Prompt-length filter ---
# Prompts longer than this many tokens are dropped BEFORE inference.
# This prevents vLLM errors (it raises, not silently truncates, when
# prompt + max_new_tokens > MAX_MODEL_LEN).  Set to MAX_MODEL_LEN minus
# the largest generation budget to leave full headroom.
MAX_PROMPT_TOKENS      = 4096   # adjustable; dropped rows are reported

# --- Evaluation tolerances ---
NUMERIC_TOL = 0.01              # 1% relative tolerance for exact match

# --- Generation parameters ---
GEN_TEMPERATURE    = 0.0
GEN_TOP_P          = 1.0
MAX_TOKENS_DIRECT  = 512        # token budget for direct (no-CoT) inference
MAX_TOKENS_COT     = 1024        # token budget for CoT inference
MAX_TOKENS_AGENTIC = 1024        # token budget per agentic turn

# --- Batch sizes (lower these if you hit OOM) ---
BATCH_SIZE_SYNTHETIC = 32       # prompts per vLLM call for synthetic test set
BATCH_SIZE_FINQA     = 16       # prompts per vLLM call for FinQA evaluation

# --- Sample limits ---
MAX_SAMPLES_SYNTHETIC = None    # None = use all TEST_SIZE samples
MAX_SAMPLES_FINQA     = 300     # FinQA cap (≤ 300)

# --- Baseline inference cache ---
# When True, base-model (no adapter) predictions are written to disk after the
# first run.  Re-runs with a different adapter load from cache instead of
# re-running baseline inference, cutting total compute time roughly in half.
# Cache is automatically invalidated when MODEL_ID or the prompt set changes.
USE_BASELINE_CACHE  = True
BASELINE_CACHE_DIR  = f"{OUTPUTS_DIR}/baseline_cache"

# --- FinQA dataset ---
# Both ibm/finqa and ibm-research/finqa use a .py loading script which is
# unsupported in datasets >= 3.0.  We load directly from the official GitHub
# repo: https://github.com/czyssrs/FinQA
FINQA_GITHUB_BASE  = "https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/FinQA"
FINQA_SPLIT        = "test"     # "test" | "validation" | "train"

print("Configuration loaded.")
print(f"  MODEL_ID:          {MODEL_ID}")
print(f"  ADAPTER_PATH:      {ADAPTER_PATH}")
print(f"  PROCESSED_DATA_DIR: {PROCESSED_DATA_DIR}")
print(f"  TEST_SIZE:         {TEST_SIZE}  (from FINAL_SAMPLE_SIZE={FINAL_SAMPLE_SIZE})")
print(f"  FINQA:             github/{FINQA_SPLIT} (max {MAX_SAMPLES_FINQA})")
print(f"  BASELINE_CACHE:    {'enabled' if USE_BASELINE_CACHE else 'disabled'}  ({BASELINE_CACHE_DIR})")

---
## Step 2 — Environment Setup

In [ ]:
import os, sys, gc, json, re, random
from pathlib import Path
from datetime import datetime

from google.colab import drive
drive.mount("/content/drive")

os.makedirs(DRIVE_BASE,  exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Clone or pull the project repo
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo exists at {REPO_DIR} — pulling latest...")
    os.system(f"cd '{REPO_DIR}' && git pull")
else:
    print(f"Cloning repo to {REPO_DIR}...")
    os.system(f"git clone '{REPO_URL}' '{REPO_DIR}'")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory : {os.getcwd()}")
print(f"sys.path[0]       : {sys.path[0]}")

---
## Step 3 — Imports

In [ ]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

from src.eval.evaluate import (
    compute_exact_match,
    compute_f1_for_task,
    is_answer_parsable,
    compute_grounding_rate,
)
from src.inference.generate import (
    build_prompt,
    _build_messages,
    _apply_chat_template,
    extract_answer_from_cot_output,
)
from src.tools.tool_router import parse_tool_call_from_output, dispatch_tool_call
from tools.financial_tools import FINANCIAL_TOOLS

from src.data.preprocess import (
    load_eval_test_samples,
    prepare_fincot_sft_dataset,
    print_preparation_summary,
)

print("Imports OK.")

---
## Step 4 — Load vLLM Model

In [ ]:
import os

# Sanity check: adapter path must exist for fine-tuned eval
if not os.path.isdir(ADAPTER_PATH):
    raise FileNotFoundError(
        f"Adapter not found at: {ADAPTER_PATH}\n"
        "Run FinReasoningAI_Colab.ipynb (SFT training) first, "
        "then re-run this notebook."
    )

torch.cuda.empty_cache()
gc.collect()

# Colab streams stdout/stderr through a pipe — vLLM needs a real fileno
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

if "MODEL" not in dir() or MODEL is None:
    print(f"Loading {MODEL_ID} with vLLM (enable_lora=True, max_lora_rank={LORA_RANK})...")
    MODEL = LLM(
        model=MODEL_ID,
        enable_lora=True,
        max_lora_rank=LORA_RANK,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        trust_remote_code=True,
        max_model_len=MAX_MODEL_LEN,
        dtype="bfloat16",
    )
    print("vLLM engine ready.")
else:
    print("MODEL already loaded — reusing existing engine.")

# Load tokenizer separately (needed for chat-template formatting)
if "TOKENIZER" not in dir() or TOKENIZER is None:
    TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if TOKENIZER.pad_token is None:
        TOKENIZER.pad_token = TOKENIZER.eos_token
    print("Tokenizer loaded.")

# LoRARequest for the fine-tuned adapter
FIN_LORA = LoRARequest("fin_adapter", 1, ADAPTER_PATH)
print(f"LoRARequest configured: {ADAPTER_PATH}")

---
## Step 5 — Shared Utilities
Batch generation helper and scoring functions reused throughout all sections.

In [ ]:
# ── Sampling params factory ────────────────────────────────────────────────────

def make_sampling_params(
    max_tokens: int,
    temperature: float = GEN_TEMPERATURE,
    top_p: float = GEN_TOP_P,
) -> SamplingParams:
    return SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        stop=["<|im_end|>", "<|endoftext|>"],
    )


# ── Batched vLLM generation ────────────────────────────────────────────────────

def batch_generate(
    prompts: list,
    *,
    lora_request=None,
    max_tokens: int = MAX_TOKENS_DIRECT,
    batch_size: int = BATCH_SIZE_SYNTHETIC,
    label: str = "",
) -> list:
    """Generate completions for all prompts in mini-batches.

    Args:
        prompts:       List of prompt strings.
        lora_request:  LoRARequest for fine-tuned model, or None for base model.
        max_tokens:    Token budget per completion.
        batch_size:    How many prompts per vLLM call (adjust for GPU memory).
        label:         Optional tqdm description.
    Returns:
        List of completion strings aligned with the input prompts.
    """
    sp = make_sampling_params(max_tokens)
    results = []
    desc = f"Generating{' ' + label if label else ''}"
    for i in tqdm(range(0, len(prompts), batch_size), desc=desc):
        batch   = prompts[i : i + batch_size]
        outputs = MODEL.generate(batch, sampling_params=sp, lora_request=lora_request)
        results.extend(o.outputs[0].text.strip() for o in outputs)
    return results


# ── Row-level scoring ─────────────────────────────────────────────────────────

def score_rows(rows: list, pred_key: str) -> dict:
    """Score each row in-place and return aggregate metrics.

    Adds ``{pred_key}_em``, ``{pred_key}_f1``, ``{pred_key}_parsable``,
    and ``{pred_key}_grounding`` to each row dict.
    """
    ems, f1s, pars, grs = [], [], [], []
    for r in rows:
        pred      = str(r.get(pred_key, ""))
        gt        = str(r.get("ground_truth", ""))
        task      = r.get("task", "financial_qa")
        ctx       = str(r.get("context", ""))
        expr      = r.get("expression")
        variables = r.get("variables") if isinstance(r.get("variables"), dict) else {}

        em       = compute_exact_match(pred, gt, tol=NUMERIC_TOL)
        f1, _    = compute_f1_for_task(task, pred, gt, em)
        parsable = is_answer_parsable(pred)
        grounding = compute_grounding_rate(
            pred, ctx,
            task=task,
            expression=str(expr) if expr is not None else None,
            variables=variables,
            numeric_tolerance=NUMERIC_TOL,
        )

        r[f"{pred_key}_em"]       = em
        r[f"{pred_key}_f1"]       = f1
        r[f"{pred_key}_parsable"] = bool(parsable)
        r[f"{pred_key}_grounding"]= grounding

        ems.append(em);  f1s.append(f1)
        pars.append(float(parsable));  grs.append(grounding)

    n = len(rows)
    return {
        "exact_match":      sum(ems) / n if n else 0.0,
        "f1":               sum(f1s) / n if n else 0.0,
        "parsability_rate": sum(pars) / n if n else 0.0,
        "grounding_rate":   sum(grs) / n if n else 0.0,
        "n_samples":        n,
    }


# ── Pretty-print helpers ──────────────────────────────────────────────────────

def print_metrics(label: str, m: dict) -> None:
    w = max(len(label), 52)
    print(f"\n{'=' * w}")
    print(f"  {label}")
    print(f"{'=' * w}")
    for k, v in m.items():
        print(f"  {k:<24}: {v:.4f}" if isinstance(v, float) else f"  {k:<24}: {v}")


def compare_metrics(base_label: str, base: dict, ft_label: str, ft: dict) -> dict:
    """Print a side-by-side comparison and return deltas."""
    keys = ("exact_match", "f1", "parsability_rate", "grounding_rate")
    delta = {k: ft.get(k, 0.0) - base.get(k, 0.0) for k in keys}
    col_w = max(len(base_label), len(ft_label), 12)
    print(f"\n  {'Metric':<24} {base_label:>{col_w}} {ft_label:>{col_w}} {'Delta':>8}")
    print("  " + "-" * (24 + 2 * col_w + 10))
    for k in keys:
        b, f, d = base.get(k, 0.0), ft.get(k, 0.0), delta[k]
        print(f"  {k:<24} {b:>{col_w}.4f} {f:>{col_w}.4f} {d:>+8.4f}")
    return delta


# ── Prompt-length helpers ─────────────────────────────────────────────────────

def _count_prompt_tokens(prompt: str) -> int:
    """Count tokens in a prompt string using the loaded TOKENIZER."""
    return len(TOKENIZER.encode(prompt, add_special_tokens=False))


def filter_by_prompt_length(
    rows: list,
    prompt_key: str,
    max_tokens: int = MAX_PROMPT_TOKENS,
    label: str = "",
) -> list:
    """Drop rows whose prompt exceeds max_tokens and report the count.

    Args:
        rows:        List of row dicts.
        prompt_key:  Key in each row dict that holds the prompt string.
                     For rows with multiple prompts (e.g. FinQA direct+CoT),
                     pass the key for the *longest* variant (usually CoT).
        max_tokens:  Hard ceiling in tokens (default MAX_PROMPT_TOKENS).
        label:       Dataset label used in the printed summary.
    Returns:
        Filtered list; original list is not modified.
    """
    kept, dropped = [], []
    for r in rows:
        n = _count_prompt_tokens(r[prompt_key])
        r[f"_n_tokens_{prompt_key}"] = n   # cache so audit cell doesn't re-tokenise
        if n <= max_tokens:
            kept.append(r)
        else:
            dropped.append(r)

    tag = f" [{label}]" if label else ""
    print(f"Prompt-length filter{tag}  (limit={max_tokens} tokens):")
    print(f"  kept   : {len(kept)}")
    print(f"  dropped: {len(dropped)}"
          + (f"  (ids: {[r.get('id','?') for r in dropped[:5]]}{'…' if len(dropped)>5 else ''})"
             if dropped else ""))
    return kept


# ── Baseline inference cache ──────────────────────────────────────────────────
# The cache is keyed by MODEL_ID + full prompt contents (SHA-256, 16 hex chars).
# Changing the model, the dataset split, or any prompt config auto-invalidates.

import hashlib as _hashlib


def _cache_key(label: str, prompts: list) -> str:
    """Return a deterministic hex key for this (model, prompt-set) pair."""
    payload = MODEL_ID + label + "".join(prompts)
    return _hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def _cache_path(label: str, prompts: list) -> str:
    key = _cache_key(label, prompts)
    os.makedirs(BASELINE_CACHE_DIR, exist_ok=True)
    return os.path.join(BASELINE_CACHE_DIR, f"baseline_{label}_{key}.json")


def load_baseline_cache(label: str, prompts: list):
    """Return list of cached predictions, or None on a cache miss / disabled."""
    if not USE_BASELINE_CACHE:
        return None
    path = _cache_path(label, prompts)
    if not os.path.exists(path):
        print(f"[cache MISS]  {label}")
        return None
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if len(data) != len(prompts):
        print(f"[cache STALE] {label}: cached {len(data)} vs needed {len(prompts)} — regenerating.")
        return None
    print(f"[cache HIT]   {label}  ({len(data)} preds)  ← {path}")
    return data


def save_baseline_cache(label: str, prompts: list, preds: list) -> None:
    """Write predictions to the cache file for future runs."""
    if not USE_BASELINE_CACHE:
        return
    path = _cache_path(label, prompts)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=None)
    print(f"[cache SAVED] {label}  ({len(preds)} preds)  → {path}")


print("Shared utilities defined.")

---
## Section A — Synthetic Test Set
Evaluates the model on the held-out test split of the synthetic dataset produced
by `FinReasoningAI_Colab.ipynb`. Compares the **base model** (no adapter) to the
**fine-tuned model** (with LoRA adapter).

In [ ]:
# ── A1: Load synthetic test data ──────────────────────────────────────────────
#
# load_eval_test_samples(data_dir) reads the DatasetDict saved by
# prepare_fincot_sft_dataset() during the Colab training run and returns
# the held-out "test" split as a list of dicts.
#
# If the processed directory doesn't exist yet (e.g. first run), this cell
# calls prepare_fincot_sft_dataset() to regenerate it — using the same
# FINAL_SAMPLE_SIZE / TRAIN_SIZE / TEST_SIZE / SEED values as the Colab notebook.

_processed_dir = Path(PROCESSED_DATA_DIR)

if not _processed_dir.exists():
    print(f"Processed dataset not found at '{PROCESSED_DATA_DIR}'.")
    print("Regenerating via prepare_fincot_sft_dataset() — this may take a few minutes...")
    _dataset_dict, _summary = prepare_fincot_sft_dataset(
        tokenizer=TOKENIZER,
        output_dir=PROCESSED_DATA_DIR,
        final_sample_size=FINAL_SAMPLE_SIZE,
        train_size=TRAIN_SIZE,
        test_size=TEST_SIZE,
        include_cot=INCLUDE_COT,
        seed=SEED,
    )
    print_preparation_summary(_summary, PROCESSED_DATA_DIR)
else:
    print(f"Found processed dataset at '{PROCESSED_DATA_DIR}' — loading test split.")

# load_eval_test_samples takes only data_dir; it loads the saved DatasetDict
# and returns dataset["test"].to_list()
synthetic_samples = load_eval_test_samples(data_dir=PROCESSED_DATA_DIR)

if MAX_SAMPLES_SYNTHETIC is not None:
    synthetic_samples = synthetic_samples[:MAX_SAMPLES_SYNTHETIC]

print(f"\nSynthetic test samples: {len(synthetic_samples)}")

# Show field names and category distribution
print(f"Row fields: {list(synthetic_samples[0].keys())}")
cat_dist = {}
for s in synthetic_samples:
    t = s.get("reasoning_category", s.get("task", "unknown"))
    cat_dist[t] = cat_dist.get(t, 0) + 1
print("Category distribution:", cat_dist)

In [ ]:
# ── A2: Build inference prompts ───────────────────────────────────────────────
#
# Each row from load_eval_test_samples already contains the original sample
# fields (question, answer, context, reasoning, reasoning_category, …) plus
# a "prompt" field (raw "context\n\nquestion" text) and "completion" field.
#
# For vLLM inference we must apply the chat template, so we use build_prompt()
# which wraps question + context into the Qwen ChatML format.
# The raw "prompt" field is NOT used for inference — only for reference.

synthetic_rows = []
for i, s in enumerate(synthetic_samples):
    question = str(s.get("question", "")).strip()
    context  = str(s.get("context",  s.get("financial_data", ""))).strip()

    # Chat-template-formatted prompt for vLLM (direct, no CoT)
    vllm_prompt = build_prompt(
        question=question,
        context=context,
        use_cot=False,
        tokenizer=TOKENIZER,
    )

    synthetic_rows.append({
        "idx":               i,
        "id":                s.get("id", str(i)),
        "task":              s.get("task", "financial_qa"),
        "reasoning_category": s.get("reasoning_category", ""),
        "question":          question,
        "ground_truth":      str(s.get("answer", "")).strip(),
        "context":           context,
        "expression":        s.get("expression"),
        "variables":         s.get("variables") if isinstance(s.get("variables"), dict) else {},
        # vLLM-ready prompt (chat-template applied)
        "prompt":            vllm_prompt,
        # Reference fields from the saved dataset
        "ref_prompt":        s.get("prompt", ""),       # raw "context\n\nquestion"
        "ref_completion":    s.get("completion", ""),   # expected completion
    })

print(f"Built {len(synthetic_rows)} inference prompts.")

# Drop prompts that would overflow MAX_MODEL_LEN and cause a vLLM error
synthetic_rows = filter_by_prompt_length(
    synthetic_rows, prompt_key="prompt",
    max_tokens=MAX_PROMPT_TOKENS, label="synthetic",
)
print(f"\nSynthetic rows after filter: {len(synthetic_rows)}")
print(f"\nvLLM prompt example (first 400 chars):")
print(synthetic_rows[0]["prompt"][:400])

In [ ]:
# ── A2b: Prompt-length audit (post-filter) ────────────────────────────────────
# Token counts were cached by filter_by_prompt_length; no re-tokenisation needed.

syn_lengths = [r["_n_tokens_prompt"] for r in synthetic_rows]
buckets = [256, 512, 1024, 2048, MAX_PROMPT_TOKENS]

print(f"Synthetic prompt lengths after filter (n={len(syn_lengths)}):")
print(f"  min={min(syn_lengths)}  median={sorted(syn_lengths)[len(syn_lengths)//2]}"
      f"  max={max(syn_lengths)}  mean={sum(syn_lengths)/len(syn_lengths):.0f}")
print(f"\n  {'Bucket (tokens)':<22} {'Count':>7} {'%':>7}")
prev = 0
for b in buckets:
    cnt = sum(1 for l in syn_lengths if prev < l <= b)
    print(f"  ({prev:>5}, {b:>5}]  {cnt:>7}  {100*cnt/len(syn_lengths):>6.1f}%")
    prev = b

# ── A3: Baseline — base model (no adapter) ────────────────────────────────────

print("\n=== Synthetic Test Set — Baseline (base model, no adapter) ===")
prompts_synthetic = [r["prompt"] for r in synthetic_rows]

baseline_preds_syn = load_baseline_cache("synthetic_baseline", prompts_synthetic)
if baseline_preds_syn is None:
    baseline_preds_syn = batch_generate(
        prompts_synthetic,
        lora_request=None,
        max_tokens=MAX_TOKENS_DIRECT,
        batch_size=BATCH_SIZE_SYNTHETIC,
        label="(synthetic / baseline)",
    )
    save_baseline_cache("synthetic_baseline", prompts_synthetic, baseline_preds_syn)

for r, pred in zip(synthetic_rows, baseline_preds_syn):
    r["baseline_pred"] = pred

metrics_syn_base = score_rows(synthetic_rows, "baseline_pred")
print_metrics("Baseline (no adapter) — Synthetic Test Set", metrics_syn_base)

In [ ]:
# ── A4: Fine-tuned — model with LoRA adapter ──────────────────────────────────

print("=== Synthetic Test Set — Fine-tuned (with LoRA adapter) ===")

finetuned_preds_syn = batch_generate(
    prompts_synthetic,
    lora_request=FIN_LORA,
    max_tokens=MAX_TOKENS_DIRECT,
    batch_size=BATCH_SIZE_SYNTHETIC,
    label="(synthetic / fine-tuned)",
)
for r, pred in zip(synthetic_rows, finetuned_preds_syn):
    r["finetuned_pred"] = pred

metrics_syn_ft = score_rows(synthetic_rows, "finetuned_pred")
print_metrics("Fine-tuned (LoRA) — Synthetic Test Set", metrics_syn_ft)

In [ ]:
# ── A5: Compare baseline vs fine-tuned ───────────────────────────────────────

print("\n=== Synthetic Test Set — Baseline vs Fine-tuned ===")
delta_syn = compare_metrics(
    "Baseline", metrics_syn_base,
    "Fine-tuned", metrics_syn_ft,
)

# Per-task breakdown
all_tasks = sorted({r["task"] for r in synthetic_rows})
if len(all_tasks) > 1:
    print("\nPer-task breakdown:")
    print(f"  {'Task':<30} {'N':>5}  {'Base EM':>8}  {'FT EM':>7}  {'Delta':>7}")
    print("  " + "-" * 62)
    for task in all_tasks:
        task_rows = [r for r in synthetic_rows if r["task"] == task]
        b_em = sum(r["baseline_pred_em"]  for r in task_rows) / len(task_rows)
        f_em = sum(r["finetuned_pred_em"] for r in task_rows) / len(task_rows)
        print(f"  {task:<30} {len(task_rows):>5}  {b_em:>8.3f}  {f_em:>7.3f}  {f_em-b_em:>+7.3f}")

RESULTS_SYNTHETIC = {
    "baseline":  metrics_syn_base,
    "finetuned": metrics_syn_ft,
    "delta":     delta_syn,
}
print("\n[Section A complete]")

---
## Section B — FinQA Evaluation
Loads up to `MAX_SAMPLES_FINQA` (≤ 300) samples from the public FinQA dataset and
evaluates **4 conditions**:

| # | Model | Inference mode |
|---|-------|----------------|
| 1 | Base (no adapter) | Direct (no CoT) |
| 2 | Base (no adapter) | CoT |
| 3 | Fine-tuned (LoRA) | Direct (no CoT) |
| 4 | Fine-tuned (LoRA) | CoT |

In [ ]:
# ── B1: Load FinQA dataset ────────────────────────────────────────────────────

# ── B1: Load FinQA from official GitHub repo ──────────────────────────────────
# ibm/finqa and ibm-research/finqa both use a .py loading script which is no
# longer supported by datasets >= 3.0.  The official repo ships plain JSON:
#   test.json  → FINQA_SPLIT="test"
#   dev.json   → FINQA_SPLIT="validation"
#   train.json → FINQA_SPLIT="train"
#
# Format per sample:
#   id, pre_text, post_text, table    ← context fields (top-level)
#   qa.question, qa.exe_ans           ← question and gold numeric answer
#   qa.program, qa.gold_inds          ← reasoning program and gold evidence

import urllib.request

_SPLIT_TO_FILE = {
    "test":       "test.json",
    "validation": "dev.json",
    "train":      "train.json",
}

def _load_finqa_from_github(split: str) -> list:
    fname = _SPLIT_TO_FILE.get(split, "test.json")
    url   = f"{FINQA_GITHUB_BASE}/{fname}"
    print(f"  GET {url}")
    with urllib.request.urlopen(url, timeout=60) as resp:
        return json.loads(resp.read().decode("utf-8"))

print(f"Loading FinQA '{FINQA_SPLIT}' split from GitHub ...")
try:
    finqa_dataset = _load_finqa_from_github(FINQA_SPLIT)
except Exception as e:
    print(f"  '{FINQA_SPLIT}' failed ({e}); trying 'validation'...")
    finqa_dataset = _load_finqa_from_github("validation")

s0 = finqa_dataset[0]
qa0 = s0.get("qa") or {}
print(f"\nTotal samples   : {len(finqa_dataset)}")
print(f"Top-level keys  : {list(s0.keys())}")
print(f"qa keys         : {list(qa0.keys())}")
print(f"\nExample id      : {s0.get('id', '?')}")
print(f"Example question: {qa0.get('question', '?')}")
print(f"Example exe_ans : {qa0.get('exe_ans', '?')}")
print(f"Example program : {qa0.get('program', '?')}")

In [ ]:
# ── B2: Format FinQA samples ──────────────────────────────────────────────────

def _table_to_str(table) -> str:
    """Convert a FinQA table (list of lists or HTML string) to plain text."""
    if not table:
        return ""
    if isinstance(table, str):
        # Some versions ship the table as HTML — do a naive strip
        return re.sub(r"<[^>]+>", " ", table).strip()
    rows = []
    for row in table:
        if isinstance(row, (list, tuple)):
            rows.append(" | ".join(str(c).strip() for c in row))
        else:
            rows.append(str(row).strip())
    return "\n".join(rows)


def _format_finqa_context(s: dict) -> str:
    """Concatenate pre_text + table + post_text into a single context string."""
    parts = []
    for field in ("pre_text",):
        val = s.get(field) or []
        if isinstance(val, str):
            val = [val]
        parts.extend(x.strip() for x in val if str(x).strip())

    table_str = _table_to_str(s.get("table") or [])
    if table_str.strip():
        parts.append("Table:\n" + table_str)

    for field in ("post_text",):
        val = s.get(field) or []
        if isinstance(val, str):
            val = [val]
        parts.extend(x.strip() for x in val if str(x).strip())

    return "\n\n".join(parts)


def _get_finqa_answer(s: dict) -> str:
    """Return the gold execution result from the nested qa dict.

    FinQA format:  s["qa"]["exe_ans"]  (numeric result of the reasoning program)
    Falls back to the program string if exe_ans is absent or empty.
    """
    qa  = s.get("qa") or {}
    exe = qa.get("exe_ans")
    if exe is not None and str(exe).strip() not in ("", "None", "nan", "null"):
        return str(exe).strip()
    # program is the reasoning expression (e.g. "subtract(3278, 3092)")
    return str(qa.get("program", "")).strip()


# Select samples: evenly spaced to preserve document diversity
finqa_all = list(finqa_dataset)
if MAX_SAMPLES_FINQA is not None and len(finqa_all) > MAX_SAMPLES_FINQA:
    step = len(finqa_all) // MAX_SAMPLES_FINQA
    finqa_all = finqa_all[:MAX_SAMPLES_FINQA * step : step][:MAX_SAMPLES_FINQA]

print(f"Using {len(finqa_all)} FinQA samples (cap={MAX_SAMPLES_FINQA}).")

# Build rows with both a direct prompt and a CoT prompt
finqa_rows = []
for i, s in enumerate(finqa_all):
    # question and answer live inside the nested "qa" dict
    qa  = s.get("qa") or {}
    ctx = _format_finqa_context(s)
    q   = str(qa.get("question", "")).strip()
    gt  = _get_finqa_answer(s)

    prompt_direct = build_prompt(
        question=q, context=ctx,
        use_cot=False, tokenizer=TOKENIZER,
    )
    prompt_cot = build_prompt(
        question=q, context=ctx,
        use_cot=True, tokenizer=TOKENIZER,
    )

    finqa_rows.append({
        "idx":           i,
        "id":            str(s.get("id", i)),      # top-level field per spec
        "task":          "financial_qa",
        "question":      q,
        "ground_truth":  gt,                        # qa.exe_ans (numeric result)
        "gold_program":  str(qa.get("program", "")),# reasoning program for reference
        "context":       ctx,
        "expression":    None,
        "variables":     {},
        "prompt_direct": prompt_direct,
        "prompt_cot":    prompt_cot,
    })

print(f"Built {len(finqa_rows)} FinQA rows (direct + CoT prompts).")

# Filter on the CoT prompt (always longer than direct) so the most demanding
# condition stays within MAX_PROMPT_TOKENS.  The direct prompt for any kept row
# is guaranteed to be even shorter, so both variants are safe.
finqa_rows = filter_by_prompt_length(
    finqa_rows, prompt_key="prompt_cot",
    max_tokens=MAX_PROMPT_TOKENS, label="finqa/cot",
)
print(f"\nFinQA rows after filter: {len(finqa_rows)}")
print("\nExample question :", finqa_rows[0]["question"])
print("Example answer   :", finqa_rows[0]["ground_truth"])
print("Context snippet  :", finqa_rows[0]["context"][:300])

In [ ]:
# ── B2b: FinQA prompt-length audit (post-filter) ─────────────────────────────
# CoT token counts were cached by filter_by_prompt_length; tokenise direct prompts now.

finqa_cot_lens    = [r["_n_tokens_prompt_cot"] for r in finqa_rows]
finqa_direct_lens = [_count_prompt_tokens(r["prompt_direct"]) for r in finqa_rows]
for r, n in zip(finqa_rows, finqa_direct_lens):
    r["_n_tokens_prompt_direct"] = n

buckets = [256, 512, 1024, 2048, MAX_PROMPT_TOKENS]
for tag, lens in [("Direct", finqa_direct_lens), ("CoT", finqa_cot_lens)]:
    print(f"\nFinQA {tag} prompts after filter (n={len(lens)}):")
    print(f"  min={min(lens)}  median={sorted(lens)[len(lens)//2]}"
          f"  max={max(lens)}  mean={sum(lens)/len(lens):.0f}")
    print(f"  {'Bucket':<22} {'Count':>7} {'%':>7}")
    prev = 0
    for b in buckets:
        cnt = sum(1 for l in lens if prev < l <= b)
        print(f"  ({prev:>5}, {b:>5}]  {cnt:>7}  {100*cnt/len(lens):>6.1f}%")
        prev = b

# ── B3: Run all 4 FinQA conditions ───────────────────────────────────────────
#
# Conditions evaluated in order:
#   1. Baseline  / Direct
#   2. Baseline  / CoT
#   3. Fine-tuned / Direct
#   4. Fine-tuned / CoT

direct_prompts = [r["prompt_direct"] for r in finqa_rows]
cot_prompts    = [r["prompt_cot"]    for r in finqa_rows]

print("--- Condition 1 / 4: Baseline / Direct ---")
preds_base_direct = load_baseline_cache("finqa_base_direct", direct_prompts)
if preds_base_direct is None:
    preds_base_direct = batch_generate(
        direct_prompts, lora_request=None,
        max_tokens=MAX_TOKENS_DIRECT, batch_size=BATCH_SIZE_FINQA,
        label="(FinQA base/direct)",
    )
    save_baseline_cache("finqa_base_direct", direct_prompts, preds_base_direct)

print("\n--- Condition 2 / 4: Baseline / CoT ---")
preds_base_cot = load_baseline_cache("finqa_base_cot", cot_prompts)
if preds_base_cot is None:
    preds_base_cot = batch_generate(
        cot_prompts, lora_request=None,
        max_tokens=MAX_TOKENS_COT, batch_size=BATCH_SIZE_FINQA,
        label="(FinQA base/cot)",
    )
    save_baseline_cache("finqa_base_cot", cot_prompts, preds_base_cot)

print("\n--- Condition 3 / 4: Fine-tuned / Direct ---")
preds_ft_direct = batch_generate(
    direct_prompts, lora_request=FIN_LORA,
    max_tokens=MAX_TOKENS_DIRECT, batch_size=BATCH_SIZE_FINQA,
    label="(FinQA ft/direct)",
)

print("\n--- Condition 4 / 4: Fine-tuned / CoT ---")
preds_ft_cot = batch_generate(
    cot_prompts, lora_request=FIN_LORA,
    max_tokens=MAX_TOKENS_COT, batch_size=BATCH_SIZE_FINQA,
    label="(FinQA ft/cot)",
)

# Attach predictions to rows
for r, p1, p2, p3, p4 in zip(
    finqa_rows, preds_base_direct, preds_base_cot,
    preds_ft_direct, preds_ft_cot
):
    r["base_direct_pred"] = p1
    r["base_cot_pred"]    = p2
    r["ft_direct_pred"]   = p3
    r["ft_cot_pred"]      = p4

print("\nAll 4 FinQA conditions generated.")

In [ ]:
# ── B4: Score FinQA conditions ────────────────────────────────────────────────

metrics_base_direct = score_rows(finqa_rows, "base_direct_pred")
metrics_base_cot    = score_rows(finqa_rows, "base_cot_pred")
metrics_ft_direct   = score_rows(finqa_rows, "ft_direct_pred")
metrics_ft_cot      = score_rows(finqa_rows, "ft_cot_pred")

for name, m in [
    ("Baseline   / Direct",   metrics_base_direct),
    ("Baseline   / CoT",      metrics_base_cot),
    ("Fine-tuned / Direct",   metrics_ft_direct),
    ("Fine-tuned / CoT",      metrics_ft_cot),
]:
    print_metrics(f"FinQA — {name}", m)

# Summary table
print("\n\n=== FinQA Evaluation Summary ===")
print(f"  {'Condition':<28} {'EM':>8} {'F1':>8} {'Parse':>8} {'Ground':>8} {'N':>5}")
print("  " + "-" * 65)
for name, m in [
    ("Baseline / Direct",   metrics_base_direct),
    ("Baseline / CoT",      metrics_base_cot),
    ("Fine-tuned / Direct", metrics_ft_direct),
    ("Fine-tuned / CoT",    metrics_ft_cot),
]:
    n = int(m.get("n_samples", 0))
    print(
        f"  {name:<28} {m['exact_match']:>8.4f} {m['f1']:>8.4f} "
        f"{m['parsability_rate']:>8.4f} {m['grounding_rate']:>8.4f} {n:>5}"
    )

# CoT improvement
print("\nCoT gain (fine-tuned, Direct → CoT):")
for k in ("exact_match", "f1"):
    d = metrics_ft_cot[k] - metrics_ft_direct[k]
    print(f"  {k}: {metrics_ft_direct[k]:.4f} → {metrics_ft_cot[k]:.4f} ({d:+.4f})")

# Adapter improvement (CoT)
print("\nAdapter gain (Baseline CoT → Fine-tuned CoT):")
delta_finqa_cot = compare_metrics(
    "Baseline CoT", metrics_base_cot,
    "Fine-tuned CoT", metrics_ft_cot,
)

RESULTS_FINQA = {
    "base_direct": metrics_base_direct,
    "base_cot":    metrics_base_cot,
    "ft_direct":   metrics_ft_direct,
    "ft_cot":      metrics_ft_cot,
}
print("\n[Section B complete]")

---
## Section C — Agentic Tool Evaluation on FinQA
The fine-tuned model is given access to the three financial tools defined in
`tools/financial_tools.py`:

| Tool | Description |
|------|-------------|
| `calculate_financial_ratio` | Compute a named ratio from two numbers |
| `parse_percentage` | Normalize a percentage string to a decimal |
| `compound_growth_rate` | Compute CAGR from start/end values and periods |

Each sample runs through a multi-turn loop (≤ 3 tool calls). Results are compared
against the **fine-tuned model with CoT but no tools** (same base condition as Condition 4 in Section B).

In [ ]:
# ── C1: Agentic generation helper ─────────────────────────────────────────────

def agentic_generate_one(
    question: str,
    context:  str,
    lora_request=None,
    max_rounds:  int = 3,
    max_tokens:  int = MAX_TOKENS_AGENTIC,
) -> dict:
    """Single-sample tool-augmented reasoning loop via vLLM.

    Builds a multi-turn chat with tool results injected as user messages.
    Works with both base model (lora_request=None) and fine-tuned (FIN_LORA).

    Returns:
        dict with keys: answer, tool_calls (list), n_rounds (int), full_output.
    """
    messages = _build_messages(
        question=question,
        context=context,
        use_cot=True,
        use_tools=True,
        tools=FINANCIAL_TOOLS,
    )
    sp = make_sampling_params(max_tokens, temperature=0.0)
    tool_calls  = []
    full_output = ""

    for _ in range(max_rounds + 1):
        prompt     = _apply_chat_template(TOKENIZER, messages, tools=FINANCIAL_TOOLS)
        outputs    = MODEL.generate([prompt], sampling_params=sp, lora_request=lora_request)
        raw_output = outputs[0].outputs[0].text.strip()
        full_output = raw_output

        parsed_call = parse_tool_call_from_output(raw_output)
        if parsed_call is None:
            # No tool call emitted — extract final answer
            answer = extract_answer_from_cot_output(raw_output)
            return {
                "answer":      answer,
                "tool_calls":  tool_calls,
                "n_rounds":    len(tool_calls),
                "full_output": raw_output,
            }

        tool_result = dispatch_tool_call(parsed_call["name"], parsed_call["arguments"])
        tool_calls.append({
            "name":      parsed_call["name"],
            "arguments": parsed_call["arguments"],
            "result":    tool_result,
        })
        messages.append({"role": "assistant", "content": raw_output})
        messages.append({
            "role": "user",
            "content": (
                f"Tool result for '{parsed_call['name']}': "
                f"{json.dumps(tool_result, ensure_ascii=False)}\n"
                "Use this result to produce your final answer."
            ),
        })

    # Exceeded max_rounds — extract from last output
    answer = extract_answer_from_cot_output(full_output)
    return {
        "answer":      answer,
        "tool_calls":  tool_calls,
        "n_rounds":    len(tool_calls),
        "full_output": full_output,
    }


print(f"Agentic loop helper ready.")
print(f"Available tools: {[t['function']['name'] for t in FINANCIAL_TOOLS]}")

In [ ]:
# ── C1b: Single-sample spot-check ─────────────────────────────────────────────
# Run ONE FinQA sample through all five inference modes and print the outputs
# side-by-side.  Change SPOT_CHECK_IDX to inspect a different sample.
#
#   Mode 1 — Baseline  / Direct   (no adapter, no CoT)
#   Mode 2 — Baseline  / CoT      (no adapter, with CoT)
#   Mode 3 — Fine-tuned / Direct  (LoRA adapter, no CoT)
#   Mode 4 — Fine-tuned / CoT     (LoRA adapter, with CoT)
#   Mode 5 — Fine-tuned / Agentic (LoRA adapter, CoT + tools)

SPOT_CHECK_IDX = 0          # ← change to any valid index in finqa_rows

_r  = finqa_rows[SPOT_CHECK_IDX]
_sp = make_sampling_params

SEP = "─" * 72

def _run1(prompt, lora_req, max_tok):
    """Run a single prompt and return the stripped completion."""
    outs = MODEL.generate(
        [prompt],
        sampling_params=make_sampling_params(max_tok),
        lora_request=lora_req,
    )
    return outs[0].outputs[0].text.strip()


print(SEP)
print(f"SPOT CHECK  —  sample idx={SPOT_CHECK_IDX}  id={_r['id']}")
print(SEP)
print(f"Question    : {_r['question']}")
print(f"Gold answer : {_r['ground_truth']}")
print(f"Gold program: {_r['gold_program']}")
print(f"\nContext (first 600 chars):\n{_r['context'][:600]}")

# ── Mode 1: Baseline / Direct ─────────────────────────────────────────────────
print(f"\n{SEP}")
print("MODE 1 — Baseline / Direct  (no adapter, no CoT)")
print(SEP)
_out_base_direct = _run1(_r["prompt_direct"], lora_req=None, max_tok=MAX_TOKENS_DIRECT)
print(_out_base_direct)

# ── Mode 2: Baseline / CoT ────────────────────────────────────────────────────
print(f"\n{SEP}")
print("MODE 2 — Baseline / CoT  (no adapter, with CoT)")
print(SEP)
_out_base_cot = _run1(_r["prompt_cot"], lora_req=None, max_tok=MAX_TOKENS_COT)
print(_out_base_cot)

# ── Mode 3: Fine-tuned / Direct ───────────────────────────────────────────────
print(f"\n{SEP}")
print("MODE 3 — Fine-tuned / Direct  (LoRA adapter, no CoT)")
print(SEP)
_out_ft_direct = _run1(_r["prompt_direct"], lora_req=FIN_LORA, max_tok=MAX_TOKENS_DIRECT)
print(_out_ft_direct)

# ── Mode 4: Fine-tuned / CoT ──────────────────────────────────────────────────
print(f"\n{SEP}")
print("MODE 4 — Fine-tuned / CoT  (LoRA adapter, with CoT)")
print(SEP)
_out_ft_cot = _run1(_r["prompt_cot"], lora_req=FIN_LORA, max_tok=MAX_TOKENS_COT)
print(_out_ft_cot)

# ── Mode 5: Fine-tuned / Agentic ─────────────────────────────────────────────
print(f"\n{SEP}")
print("MODE 5 — Fine-tuned / Agentic  (LoRA adapter, CoT + tools)")
print(SEP)
_agentic_result = agentic_generate_one(
    question=_r["question"],
    context=_r["context"],
    lora_request=FIN_LORA,
)
if _agentic_result["tool_calls"]:
    print(f"Tool calls ({_agentic_result['n_rounds']} round(s)):")
    for _tc in _agentic_result["tool_calls"]:
        print(f"  → {_tc['name']}({json.dumps(_tc['arguments'])})  =  {_tc['result']}")
    print()
else:
    print("(no tool calls made)\n")
print(_agentic_result["full_output"])

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print(f"{'Mode':<38} {'Extracted answer':<30} EM")
print("─" * 72)
_gold = _r["ground_truth"]
for _mode, _raw in [
    ("1  Baseline / Direct",         _out_base_direct),
    ("2  Baseline / CoT",            extract_answer_from_cot_output(_out_base_cot)),
    ("3  Fine-tuned / Direct",       _out_ft_direct),
    ("4  Fine-tuned / CoT",          extract_answer_from_cot_output(_out_ft_cot)),
    ("5  Fine-tuned / Agentic",      _agentic_result["answer"]),
]:
    _em = compute_exact_match(str(_raw), str(_gold), tol=NUMERIC_TOL)
    print(f"  {_mode:<36} {str(_raw)[:28]:<30} {'✓' if _em else '✗'}")
print(SEP)

In [ ]:
# ── C2: Run agentic evaluation ────────────────────────────────────────────────
#
# The agentic loop runs sample-by-sample (no batching) because each sample
# may require a different number of tool-call turns.

print(f"=== FinQA — Agentic Tool Evaluation (fine-tuned + tools, n={len(finqa_rows)}) ===")
print("Running turn-by-turn tool loop — this may take several minutes...\n")

finqa_agentic_rows = []
tool_usage = {}

for i, r in enumerate(tqdm(finqa_rows, desc="Agentic eval (fine-tuned)")):
    result = agentic_generate_one(
        question=r["question"],
        context=r["context"],
        lora_request=FIN_LORA,
    )
    row = dict(r)   # copy of finqa_row
    row["agentic_pred"]     = result["answer"]
    row["n_tool_rounds"]    = result["n_rounds"]
    row["tool_calls_json"]  = json.dumps(result["tool_calls"], ensure_ascii=False)
    finqa_agentic_rows.append(row)

    for tc in result["tool_calls"]:
        name = tc["name"]
        tool_usage[name] = tool_usage.get(name, 0) + 1

# Usage statistics
total_calls       = sum(tool_usage.values())
samples_w_tools   = sum(1 for r in finqa_agentic_rows if r["n_tool_rounds"] > 0)
avg_rounds_active = (
    sum(r["n_tool_rounds"] for r in finqa_agentic_rows if r["n_tool_rounds"] > 0)
    / max(samples_w_tools, 1)
)

print(f"\nTool usage summary:")
print(f"  Samples that used ≥1 tool : {samples_w_tools}/{len(finqa_rows)}"
      f" ({100*samples_w_tools/len(finqa_rows):.1f}%)")
print(f"  Total tool calls           : {total_calls}")
print(f"  Avg rounds (when tools used): {avg_rounds_active:.2f}")
for name, cnt in sorted(tool_usage.items(), key=lambda x: -x[1]):
    print(f"    {name:<35}: {cnt}")

In [ ]:
# ── C3: Score and compare agentic vs no-tools ─────────────────────────────────

metrics_ft_agentic = score_rows(finqa_agentic_rows, "agentic_pred")

print_metrics("Fine-tuned / Agentic (with tools)",  metrics_ft_agentic)
print_metrics("Fine-tuned / CoT     (no tools)",    metrics_ft_cot)

print("\n=== Fine-tuned: CoT (no tools) vs Agentic (tools) ===")
delta_agentic = compare_metrics(
    "FT / CoT (no tools)", metrics_ft_cot,
    "FT / Agentic (tools)", metrics_ft_agentic,
)

RESULTS_AGENTIC = {
    "ft_cot_no_tools":   metrics_ft_cot,
    "ft_agentic_tools":  metrics_ft_agentic,
    "delta":             delta_agentic,
    "tool_usage":        tool_usage,
    "samples_with_tools": samples_w_tools,
    "total_tool_calls":  total_calls,
}
print("\n[Section C complete]")

---
## Section D — Full Summary & Export

In [ ]:
# ── D1: Combined summary table ────────────────────────────────────────────────

_syn = globals().get("RESULTS_SYNTHETIC", {})
_fqa = globals().get("RESULTS_FINQA",    {})
_agt = globals().get("RESULTS_AGENTIC",  {})

SUMMARY_ROWS = []
if _syn:
    SUMMARY_ROWS += [
        ("Synthetic — Baseline (no adapter)",   _syn["baseline"]),
        ("Synthetic — Fine-tuned (LoRA)",        _syn["finetuned"]),
    ]
if _fqa:
    SUMMARY_ROWS += [
        ("FinQA — Baseline / Direct",            _fqa["base_direct"]),
        ("FinQA — Baseline / CoT",               _fqa["base_cot"]),
        ("FinQA — Fine-tuned / Direct",          _fqa["ft_direct"]),
        ("FinQA — Fine-tuned / CoT",             _fqa["ft_cot"]),
    ]
if _agt:
    SUMMARY_ROWS += [
        ("FinQA — Fine-tuned / Agentic (tools)", _agt["ft_agentic_tools"]),
    ]

w = 44
print("\n" + "=" * (w + 42))
print(f"  {'FULL EVALUATION SUMMARY':^{w + 38}}")
print("=" * (w + 42))
print(f"  {'Condition':<{w}} {'EM':>8} {'F1':>8} {'Parse':>8} {'Ground':>8} {'N':>6}")
print("  " + "-" * (w + 42))
for name, m in SUMMARY_ROWS:
    n = int(m.get("n_samples", 0))
    print(
        f"  {name:<{w}} {m['exact_match']:>8.4f} {m['f1']:>8.4f} "
        f"{m['parsability_rate']:>8.4f} {m['grounding_rate']:>8.4f} {n:>6}"
    )
print("=" * (w + 42))

# Key deltas
print("\nKey deltas (Exact Match):")
deltas = []
if _syn:
    deltas.append(("Synthetic: Base → FT",        _syn["delta"]["exact_match"]))
if _fqa:
    deltas.append(("FinQA FT: Direct → CoT",
                   _fqa["ft_cot"]["exact_match"] - _fqa["ft_direct"]["exact_match"]))
    deltas.append(("FinQA Base→FT (CoT)",
                   _fqa["ft_cot"]["exact_match"] - _fqa["base_cot"]["exact_match"]))
if _agt:
    deltas.append(("FinQA FT: CoT → Agentic",     _agt["delta"]["exact_match"]))
for label, d in deltas:
    print(f"  {label:<40}: {d:+.4f}")

In [ ]:
# ── D2: Export all results ────────────────────────────────────────────────────

import os
os.makedirs(OUTPUTS_DIR, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")

# --- Synthetic test set CSV (only when Section A was run) ---
_synthetic_rows = globals().get("synthetic_rows")
if _synthetic_rows:
    synthetic_export = [{
        "id":                  r["id"],
        "task":                r["task"],
        "question":            r["question"],
        "context":             r["context"][:500],
        "ground_truth":        r["ground_truth"],
        "baseline_pred":       r.get("baseline_pred", ""),
        "finetuned_pred":      r.get("finetuned_pred", ""),
        "baseline_em":         r.get("baseline_pred_em"),
        "finetuned_em":        r.get("finetuned_pred_em"),
        "baseline_f1":         r.get("baseline_pred_f1"),
        "finetuned_f1":        r.get("finetuned_pred_f1"),
        "baseline_parsable":   r.get("baseline_pred_parsable"),
        "finetuned_parsable":  r.get("finetuned_pred_parsable"),
        "baseline_grounding":  r.get("baseline_pred_grounding"),
        "finetuned_grounding": r.get("finetuned_pred_grounding"),
    } for r in _synthetic_rows]
    syn_csv = f"{OUTPUTS_DIR}/eval_synthetic_{ts}.csv"
    pd.DataFrame(synthetic_export).to_csv(syn_csv, index=False, encoding="utf-8")
    print(f"Saved: {syn_csv}  ({len(synthetic_export)} rows)")
else:
    print("Skipping synthetic CSV — Section A was not run.")

# --- FinQA full CSV ---
finqa_export = []
for r, ar in zip(finqa_rows, finqa_agentic_rows):
    finqa_export.append({
        "id":                  r["id"],
        "question":            r["question"],
        "context":             r["context"][:500],
        "ground_truth":        r["ground_truth"],   # exe_ans (numeric)
        "gold_program":        r.get("gold_program", ""),
        # Condition 1: baseline / direct
        "base_direct_pred":    r.get("base_direct_pred", ""),
        "base_direct_em":      r.get("base_direct_pred_em"),
        "base_direct_f1":      r.get("base_direct_pred_f1"),
        # Condition 2: baseline / CoT
        "base_cot_pred":       r.get("base_cot_pred", ""),
        "base_cot_em":         r.get("base_cot_pred_em"),
        "base_cot_f1":         r.get("base_cot_pred_f1"),
        # Condition 3: fine-tuned / direct
        "ft_direct_pred":      r.get("ft_direct_pred", ""),
        "ft_direct_em":        r.get("ft_direct_pred_em"),
        "ft_direct_f1":        r.get("ft_direct_pred_f1"),
        # Condition 4: fine-tuned / CoT
        "ft_cot_pred":         r.get("ft_cot_pred", ""),
        "ft_cot_em":           r.get("ft_cot_pred_em"),
        "ft_cot_f1":           r.get("ft_cot_pred_f1"),
        # Condition 5: fine-tuned / agentic
        "agentic_pred":        ar.get("agentic_pred", ""),
        "agentic_em":          ar.get("agentic_pred_em"),
        "agentic_f1":          ar.get("agentic_pred_f1"),
        "n_tool_rounds":       ar.get("n_tool_rounds"),
        "tool_calls_json":     ar.get("tool_calls_json"),
    })
finqa_csv = f"{OUTPUTS_DIR}/eval_finqa_{ts}.csv"
pd.DataFrame(finqa_export).to_csv(finqa_csv, index=False, encoding="utf-8")
print(f"Saved: {finqa_csv}  ({len(finqa_export)} rows)")

# --- Summary JSON ---
_finqa_rows_g = globals().get("finqa_rows", [])
summary_all = {
    "timestamp":     ts,
    "model_id":      MODEL_ID,
    "adapter_path":  ADAPTER_PATH,
    "config": {
        "numeric_tol":          NUMERIC_TOL,
        "gen_temperature":       GEN_TEMPERATURE,
        "max_tokens_direct":     MAX_TOKENS_DIRECT,
        "max_tokens_cot":        MAX_TOKENS_COT,
        "max_tokens_agentic":    MAX_TOKENS_AGENTIC,
        "batch_size_synthetic":  BATCH_SIZE_SYNTHETIC,
        "batch_size_finqa":      BATCH_SIZE_FINQA,
        "n_synthetic_test":      len(_synthetic_rows or []),
        "n_finqa":               len(_finqa_rows_g),
        "finqa_github_base":     FINQA_GITHUB_BASE,
        "finqa_split":           FINQA_SPLIT,
    },
    "results": {
        **({"synthetic": _syn} if _syn else {}),
        **({"finqa":     _fqa} if _fqa else {}),
        **({"agentic":   _agt} if _agt else {}),
    },
}
json_path = f"{OUTPUTS_DIR}/eval_summary_{ts}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary_all, f, ensure_ascii=False, indent=2)
print(f"Saved: {json_path}")

print("\n[All evaluation artifacts saved successfully.]")